In [0]:
# COMMAND ----------
# CELL 1: Setup & Parameters
import re
import json
import urllib.request
import urllib.error
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType, DoubleType, BooleanType
)
from sodapy import Socrata
import json

dbutils.widgets.text("domain", "opendata.maryland.gov", "Socrata Domain")
dbutils.widgets.text("dataset_id", "", "Single Dataset ID")
dbutils.widgets.text("catalog_name", "open_data_portal", "Catalog Name")
dbutils.widgets.text("schema_name", "bronze", "Target Schema Name")
dbutils.widgets.text("meta_schema", "meta", "Control Schema Name")
dbutils.widgets.dropdown("verbose", "true", ["true", "false"], "Verbose Logging")
dbutils.widgets.text("portal_data_updated_at", "", "Portal Data Updated At")

SOCRATA_DOMAIN = dbutils.widgets.get("socrata_domain")
DATASET_ID = dbutils.widgets.get("dataset_id").strip()
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
META_SCHEMA = dbutils.widgets.get("meta_schema")
VERBOSE = dbutils.widgets.get("verbose").lower() == "true"
PORTAL_DATA_UPDATED_AT = dbutils.widgets.get("portal_data_updated_at").strip()

QUEUE_TABLE = f"`{CATALOG_NAME}`.`{META_SCHEMA}`.`_socrata_ingestion_queue`"

if not DATASET_ID:
    raise ValueError("dataset_id widget must be provided for single-task ingestion.")

def log(*args, **kwargs):
    if VERBOSE:
        print(*args, **kwargs)

def sanitize_table_name(name: str) -> str:
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', (name or '').lower())
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean[:128]

def sql_str(val) -> str:
    if val is None:
        return ""
    return str(val).strip().replace("\\", "\\\\").replace("'", "\\'")

def run_sql(label: str, stmt: str) -> bool:
    try:
        spark.sql(stmt)
        log(f"    [SUCCESS] {label}")
        return True
    except Exception as err:
        log(f"    [WARNING] {label} failed: {str(err).splitlines()[0]}")
        return False

def append_log_event(
    dataset_id: str, 
    title: str, 
    table_name: str, 
    category: str, 
    portal_data_update: str, 
    status: str, 
    records: int = 0, 
    error: str = ""
):
    """
    Appends an immutable audit row to the queue table with task_type='DATA'.
    Leaves portal_metadata_updated_at as None.
    """
    schema = StructType([
        StructField("dataset_id", StringType(), False),
        StructField("task_type", StringType(), True),
        StructField("title", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("category", StringType(), True),
        StructField("portal_data_updated_at", StringType(), True),
        StructField("portal_metadata_updated_at", StringType(), True),
        StructField("status", StringType(), True),
        StructField("records_ingested", LongType(), True),
        StructField("error_message", StringType(), True)
    ])

    resolved_data_update = portal_data_update or PORTAL_DATA_UPDATED_AT or None

    log_row = [{
        "dataset_id": dataset_id,
        "task_type": "DATA",
        "title": title,
        "table_name": table_name,
        "category": category,
        "portal_data_updated_at": resolved_data_update,
        "portal_metadata_updated_at": None,
        "status": status,
        "records_ingested": records,
        "error_message": str(error)[:1000] if error else None
    }]

    (spark.createDataFrame(log_row, schema=schema)
        .withColumn("logged_at", current_timestamp())
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(QUEUE_TABLE))

In [0]:
# COMMAND ----------
# CELL 2: Ingest, Transform, and Save Single Unified Table
from requests.exceptions import RequestException, HTTPError
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception
)

def is_retryable_error(exception: BaseException) -> bool:
    if isinstance(exception, urllib.error.HTTPError):
        return exception.code in (429, 500, 502, 503, 504)
    if isinstance(exception, HTTPError) and exception.response is not None:
        return exception.response.status_code in (429, 500, 502, 503, 504)
    if isinstance(exception, (RequestException, urllib.error.URLError, TimeoutError, ConnectionError)):
        return True
    return False

soda_retry = retry(
    reraise=True,
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=2, max=60),
    retry=retry_if_exception(is_retryable_error),
    before_sleep=lambda rs: log(
        f"    [RETRY] Attempt {rs.attempt_number} failed ({rs.outcome.exception()}). "
        f"Retrying in {rs.next_action.sleep:.2f}s..."
    )
)

@soda_retry
def fetch_metadata_with_retry(soda_client, d_id):
    return soda_client.get_metadata(d_id)

@soda_retry
def fetch_page_with_retry(soda_client, d_id, limit, offset, order=":id"):
    try:
        return soda_client.get(d_id, limit=limit, offset=offset, order=order)
    except Exception:
        return soda_client.get(d_id, limit=limit, offset=offset)

SECRET_SCOPE = "socrata_scope"
try:
    APP_TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key="app_token")
    log("Loaded Socrata app token from secret scope.")
except Exception:
    APP_TOKEN = None
    log("Warning: Running unauthenticated.")

client = Socrata(SOCRATA_DOMAIN, APP_TOKEN, timeout=600)

MAX_TAGS = 40
GEO_TYPES = {
    'point', 'multipoint', 'linestring', 'multilinestring', 
    'polygon', 'multipolygon', 'location', 'geometry'
}

def sanitize_tag_key(key: str) -> str:
    clean = re.sub(r'[^a-zA-Z0-9_\-]', '_', str(key).strip().lower())
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean[:255]

log(f"\nProcessing Dataset: {DATASET_ID}")

# 1. Fetch Upstream Metadata & Validate Public Status
try:
    dataset_metadata = fetch_metadata_with_retry(client, DATASET_ID)
except urllib.error.HTTPError as http_err:
    if http_err.code in (403, 404):
        append_log_event(DATASET_ID, DATASET_ID, "Uncategorized", None, "DEPRECATED", 0, f"HTTP {http_err.code}")
        dbutils.notebook.exit(f"DEPRECATED_HTTP_{http_err.code}")
    raise http_err
except Exception as err:
    append_log_event(DATASET_ID, DATASET_ID, "Uncategorized", None, "FAILED", 0, str(err))
    raise RuntimeError(f"Metadata fetch failed: {err}")

dataset_title = dataset_metadata.get('name', DATASET_ID)
category = dataset_metadata.get("category") or "Uncategorized"
portal_updated_at = (dataset_metadata.get("rowsUpdatedAt") or dataset_metadata.get("dataUpdatedAt"))

approvals = (dataset_metadata.get("approvals") or [{}])[0] or {}
target_audience = (approvals.get("targetAudience") or "").lower()
publication_stage = (dataset_metadata.get("publicationStage") or "").lower()

if not (publication_stage == "published" and target_audience == "public"):
    append_log_event(DATASET_ID, dataset_title, category, portal_updated_at, "DEPRECATED", 0, "Not public/published")
    dbutils.notebook.exit("Filtered non-public dataset.")

table_name = sanitize_table_name(dataset_title)
target_table_path = f"`{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{table_name}`"

# 2. Schema Discovery from Socrata Column Definitions
socrata_columns = dataset_metadata.get('columns', [])
all_meta_columns = [c.get('fieldName') for c in socrata_columns if c.get('fieldName')]

geo_cols = set()
socrata_type_cast_map = {}

for c in socrata_columns:
    fn = c.get('fieldName')
    dt = (c.get('dataTypeName') or '').lower()
    if not fn:
        continue
    if dt in GEO_TYPES:
        geo_cols.add(fn)
    elif dt in ['number', 'money']:
        socrata_type_cast_map[fn] = 'double'
    elif dt == 'checkbox':
        socrata_type_cast_map[fn] = 'boolean'

serverless_input_schema = StructType([
    StructField(col_name, StringType(), True) for col_name in all_meta_columns
])
cast_exprs = {c: col(c).cast(t) for c, t in socrata_type_cast_map.items()}

# ==============================================================================
# 3. Stream SODA Rows Directly to Disk (Unity Catalog Volume / Staging Path)
# ==============================================================================
import os
import uuid
import shutil

# Staging path in UC Volume (or use f"/tmp/socrata_staging/{DATASET_ID}_{uuid.uuid4().hex[:8]}" on single node)
STAGING_DIR = f"/Volumes/{CATALOG_NAME}/{META_SCHEMA}/socrata_staging/{DATASET_ID}_{uuid.uuid4().hex[:8]}"
os.makedirs(STAGING_DIR, exist_ok=True)

offset = 0
BATCH_SIZE = 50000 if len(all_meta_columns) <= 25 else 20000
total_records = 0
batch_index = 0

try:
    while True:
        records = fetch_page_with_retry(client, DATASET_ID, limit=BATCH_SIZE, offset=offset, order=":id")
        if not records:
            break

        chunk_path = os.path.join(STAGING_DIR, f"part_{batch_index:05d}.jsonl")
        
        # Stream batch directly to a newline-delimited JSON file
        with open(chunk_path, "w", encoding="utf-8") as f_out:
            for r in records:
                row_dict = {}
                for col_name in all_meta_columns:
                    val = r.get(col_name)
                    if val is None:
                        row_dict[col_name] = None
                    elif isinstance(val, (dict, list)):
                        row_dict[col_name] = json.dumps(val)
                    else:
                        val_str = str(val).strip()
                        if col_name in socrata_type_cast_map and socrata_type_cast_map[col_name] == 'double':
                            val_str = re.sub(r'[\$,]', '', val_str)
                        row_dict[col_name] = val_str if val_str else None
                
                f_out.write(json.dumps(row_dict) + "\n")

        total_records += len(records)
        batch_index += 1
        offset += BATCH_SIZE
        
        log(f"    Staged batch {batch_index}: {len(records)} rows written (Total: {total_records}).")
        
        if len(records) < BATCH_SIZE:
            break

    if total_records == 0:
        if os.path.exists(STAGING_DIR):
            shutil.rmtree(STAGING_DIR)
        append_log_event(DATASET_ID, dataset_title, category, portal_updated_at, "EMPTY", 0, None)
        dbutils.notebook.exit("EMPTY")

    # ==============================================================================
    # 4. Read Distributed Dataset via Spark (Driver Memory Protected)
    # ==============================================================================
    df = (
        spark.read
        .schema(serverless_input_schema)
        .json(STAGING_DIR)
    )

    if cast_exprs:
        df = df.withColumns(cast_exprs)

    loc_col = next((c for c in geo_cols if c in df.columns), None)
    if not loc_col:
        lower_cols = {c.lower(): c for c in df.columns}
        loc_col = next((lower_cols[c] for c in ["location", "location_1", "the_geom", "geom", "geometry"] if c in lower_cols), None)

    sample_val = ""
    if loc_col:
        sample_row = df.select(loc_col).filter(col(loc_col).isNotNull() & (F.trim(col(loc_col)) != "")).limit(1).collect()
        sample_val = str(sample_row[0][loc_col]).strip() if sample_row else ""

    has_geometry = False

    # Check for Socrata JSON (nested human_address or coordinates)
    if loc_col and ("human_address" in sample_val or "latitude" in sample_val):
        df = (
            df.withColumn("street_address", F.col(loc_col).cast("string"))
              .withColumn("street_address", F.get_json_object(F.get_json_object(col(loc_col), "$.human_address"), "$.address"))
              .withColumn("city", F.get_json_object(F.get_json_object(col(loc_col), "$.human_address"), "$.city"))
              .withColumn("state", F.get_json_object(F.get_json_object(col(loc_col), "$.human_address"), "$.state"))
              .withColumn("zip", F.expr(f"cast(get_json_object(get_json_object({loc_col}, '$.human_address'), '$.zip') AS STRING)"))
              .withColumn("__lat", F.expr(f"try_cast(get_json_object({loc_col}, '$.latitude') AS DOUBLE)"))
              .withColumn("__lon", F.expr(f"try_cast(get_json_object({loc_col}, '$.longitude') AS DOUBLE)"))
        )
        has_coords = df.filter(col("__lat").isNotNull() & col("__lon").isNotNull()).limit(1).count() > 0
        if has_coords:
            has_geometry = True
            df = df.withColumn(
                "__g",
                F.expr("""
                    CASE 
                        WHEN __lon BETWEEN -180 AND 180 AND __lat BETWEEN -90 AND 90 AND NOT (__lon = 0 AND __lat = 0)
                        THEN st_point(__lon, __lat, 4326)
                        ELSE NULL 
                    END
                """)
            )

    # Check for separate numeric lat/lon columns
    elif "latitude" in [c.lower() for c in df.columns] and "longitude" in [c.lower() for c in df.columns]:
        l_map = {c.lower(): c for c in df.columns}
        lat_c, lon_c = l_map["latitude"], l_map["longitude"]
        df = (
            df.withColumn("__lat", F.expr(f"try_cast({lat_c} AS DOUBLE)"))
              .withColumn("__lon", F.expr(f"try_cast({lon_c} AS DOUBLE)"))
        )
        has_coords = df.filter(col("__lat").isNotNull() & col("__lon").isNotNull()).limit(1).count() > 0
        if has_coords:
            has_geometry = True
            df = df.withColumn(
                "__g",
                F.expr("""
                    CASE 
                        WHEN __lon BETWEEN -180 AND 180 AND __lat BETWEEN -90 AND 90 AND NOT (__lon = 0 AND __lat = 0)
                        THEN st_point(__lon, __lat, 4326)
                        ELSE NULL 
                    END
                """)
            )

    # Check for GeoJSON string
    elif loc_col and '"coordinates"' in sample_val:
        has_geometry = True
        df = df.withColumn(
            "__g",
            F.expr(f"CASE WHEN {loc_col} IS NOT NULL AND length(trim({loc_col})) > 0 THEN st_setsrid(st_geomfromgeojson({loc_col}), 4326) ELSE NULL END")
        )

    # Ensure any zip column is explicitly cast to String
    for z_candidate in ["zip", "zip_code", "zipcode", "postal_code"]:
        if z_candidate in df.columns:
            df = df.withColumn(z_candidate, col(z_candidate).cast("string"))

    # Spatial column enrichment
    if has_geometry:
        df = (
            df.withColumn("geom", F.expr("__g"))
              .withColumn("geom_md", F.expr("st_transform(__g, 6487)"))
              .withColumn(
                  "h3_r8",
                  F.expr("""
                      CASE 
                          WHEN geom IS NULL OR st_isempty(geom) THEN NULL
                          WHEN abs(st_x(st_centroid(geom))) > 180 OR abs(st_y(st_centroid(geom))) > 90 THEN NULL
                          ELSE h3_longlatash3(st_x(st_centroid(geom)), st_y(st_centroid(geom)), 8)
                      END
                  """)
              )
        )

    drop_candidates = ["__g", "__lat", "__lon"]
    if loc_col:
        drop_candidates.append(loc_col)

    cols_to_drop = [
        c for c in df.columns 
        if c in drop_candidates 
        or c.startswith("@computed_region") 
        or c.startswith(":")
    ]
    df = df.drop(*cols_to_drop)

# ==============================================================================
    # 5. Commit to Delta Lake & Clean Up Disk Staging
    # ==============================================================================
    clean_target = target_table_path.replace("`", "")
    if spark.catalog.tableExists(clean_target):
        if not (has_geometry and "h3_r8" in df.columns):
            try:
                spark.sql(f"ALTER TABLE {target_table_path} UNSET TBLPROPERTIES IF EXISTS ('delta.dataSkippingStatsColumns')")
                log(f"Pre-write: Removed 'delta.dataSkippingStatsColumns' on non-spatial table {target_table_path}.")
            except Exception as e:
                log(f"Pre-write property cleanup warning: {e}")

    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(target_table_path)
    )
    log(f"Committed unified table: {target_table_path} ({total_records} rows).")

    # ==============================================================================
    # 5b. Update Table Metadata ONLY for data_updated_at
    # ==============================================================================
    resolved_data_ts = portal_updated_at or PORTAL_DATA_UPDATED_AT
    if resolved_data_ts:
        clean_ts = sql_str(str(resolved_data_ts))
        try:
            spark.sql(f"""
                ALTER TABLE {target_table_path} 
                SET TBLPROPERTIES ('open_data.data_updated_at' = '{clean_ts}')
            """)
            log(f"Updated TBLPROPERTIES 'open_data.data_updated_at' = '{clean_ts}' on {target_table_path}.")
        except Exception as prop_err:
            log(f"Warning: Failed to set table property 'open_data.data_updated_at': {prop_err}")

    # ==============================================================================
    # 6. Log Success
    # ==============================================================================
    append_log_event(
        dataset_id=DATASET_ID,
        title=dataset_title,
        table_name=table_name,
        category=category,
        portal_data_update=resolved_data_ts,
        status="SUCCESS",
        records=total_records,
        error=None
    )

except Exception as pipeline_err:
    append_log_event(
        dataset_id=DATASET_ID,
        title=locals().get("dataset_title", DATASET_ID),
        table_name=locals().get("table_name", DATASET_ID),
        category=locals().get("category", "Uncategorized"),
        portal_data_update=locals().get("portal_updated_at", PORTAL_DATA_UPDATED_AT),
        status="FAILED",
        records=0,
        error=str(pipeline_err)
    )
    raise RuntimeError(f"Ingestion failed for {DATASET_ID}: {pipeline_err}")

finally:
    # Always clean up disk staging directory whether success or failure
    if os.path.exists(STAGING_DIR):
        try:
            shutil.rmtree(STAGING_DIR)
            log(f"Cleaned up staging folder: {STAGING_DIR}")
        except Exception as cleanup_err:
            log(f"Warning: Failed to clean up staging folder: {cleanup_err}")